# Customer Segment Flow Analysis - Sankey Visualization

## Objective
Visualize customer segment transitions year-over-year using Sankey diagrams to understand:
- How customers move between segments (Active, At Risk, Inactive, Dormant)
- Segment retention and churn patterns
- Growth of customer base over time

## Use Case
This analysis was previously done in Power BI using DAX. Now implemented in Python for portfolio showcase.

## Methodology
1. Load yearly snapshot data (Python equivalent of Power BI YearlySnapshot)
2. Calculate segment transitions between consecutive years
3. Create interactive Sankey diagrams showing flow
4. Analyze segment retention and migration patterns

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully!")

## 1. Load Yearly Snapshot Data

In [ ]:
# Load yearly snapshot (equivalent to Power BI YearlySnapshot table)
snapshot_df = pd.read_csv('../data/yearly_snapshot.csv')

# Convert date columns
date_columns = ['MaxSaleDate', 'FirstPurchaseDate', 'Year_End', 'Cumulative_Last_Txn']
for col in date_columns:
    if col in snapshot_df.columns:
        snapshot_df[col] = pd.to_datetime(snapshot_df[col])

print(f"Loaded {len(snapshot_df):,} records")
print(f"Years: {sorted(snapshot_df['YEAR'].unique())}")
print(f"Members: {snapshot_df['Member_ID'].nunique():,}")

display(snapshot_df.head())

## 2. Segment Distribution by Year

In [ ]:
# Get segment distribution by year
segment_by_year = snapshot_df.groupby(['YEAR', 'Recency_Text']).agg({
    'Member_ID': 'count',
    'TotalSpend': 'sum'
}).reset_index()

segment_by_year.columns = ['Year', 'Segment', 'Customer_Count', 'Total_Revenue']

print("\n=== Segment Distribution by Year ===")
pivot_table = segment_by_year.pivot(index='Year', columns='Segment', values='Customer_Count')
display(pivot_table)

# Visualize segment distribution over time
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Customer count by segment
pivot_table.plot(kind='bar', stacked=True, ax=axes[0], 
                 color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6C757D'])
axes[0].set_title('Customer Count by Segment (Year-over-Year)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Customers')
axes[0].legend(title='Segment', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].tick_params(axis='x', rotation=0)

# Revenue by segment
revenue_pivot = segment_by_year.pivot(index='Year', columns='Segment', values='Total_Revenue')
revenue_pivot.plot(kind='bar', stacked=True, ax=axes[1],
                   color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6C757D'])
axes[1].set_title('Revenue by Segment (Year-over-Year)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Revenue ($)')
axes[1].legend(title='Segment', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../visualizations/segment_distribution_yearly.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Calculate Segment Transitions

Track how customers move from one segment to another between consecutive years.

In [ ]:
def calculate_segment_transitions(df, year1, year2):
    """
    Calculate segment transitions between two consecutive years.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Yearly snapshot dataframe
    year1 : str or int
        Starting year
    year2 : str or int
        Ending year
        
    Returns:
    --------
    pd.DataFrame
        Transition counts from year1 to year2
    """
    # Get data for both years
    year1_data = df[df['YEAR'] == str(year1)][['Member_ID', 'Recency_Text']].copy()
    year2_data = df[df['YEAR'] == str(year2)][['Member_ID', 'Recency_Text']].copy()
    
    # Rename columns for clarity
    year1_data.columns = ['Member_ID', 'Segment_From']
    year2_data.columns = ['Member_ID', 'Segment_To']
    
    # Merge to get transitions
    transitions = year1_data.merge(year2_data, on='Member_ID', how='inner')
    
    # Count transitions
    transition_counts = transitions.groupby(['Segment_From', 'Segment_To']).size().reset_index()
    transition_counts.columns = ['Source', 'Target', 'Count']
    
    # Add year labels to segments
    transition_counts['Source'] = transition_counts['Source'] + f' ({year1})'
    transition_counts['Target'] = transition_counts['Target'] + f' ({year2})'
    
    return transition_counts


# Calculate transitions for all consecutive year pairs
years = sorted([int(y) for y in snapshot_df['YEAR'].unique()])
all_transitions = []

for i in range(len(years) - 1):
    year1, year2 = years[i], years[i + 1]
    transitions = calculate_segment_transitions(snapshot_df, year1, year2)
    transitions['Year_Pair'] = f'{year1}-{year2}'
    all_transitions.append(transitions)

# Combine all transitions
all_transitions_df = pd.concat(all_transitions, ignore_index=True)

print("\n=== Sample Transitions ===")
display(all_transitions_df.head(20))

print(f"\nTotal transition records: {len(all_transitions_df):,}")

## 4. Sankey Diagram - Individual Year Transitions

Create Sankey diagrams for each year-to-year transition.

In [ ]:
def create_sankey_diagram(transitions_df, title="Customer Segment Flow"):
    """
    Create a Sankey diagram from transition data.
    
    Parameters:
    -----------
    transitions_df : pd.DataFrame
        DataFrame with Source, Target, and Count columns
    title : str
        Title for the diagram
        
    Returns:
    --------
    plotly.graph_objects.Figure
        Interactive Sankey diagram
    """
    # Get unique nodes (segments)
    nodes = list(set(transitions_df['Source'].tolist() + transitions_df['Target'].tolist()))
    node_dict = {node: idx for idx, node in enumerate(nodes)}
    
    # Map source and target to indices
    transitions_df['Source_idx'] = transitions_df['Source'].map(node_dict)
    transitions_df['Target_idx'] = transitions_df['Target'].map(node_dict)
    
    # Define colors for segments
    segment_colors = {
        'Active': 'rgba(46, 134, 171, 0.8)',  # Blue
        'At Risk': 'rgba(241, 143, 1, 0.8)',  # Orange
        'Inactive': 'rgba(162, 59, 114, 0.8)',  # Purple
        'Dormant': 'rgba(199, 62, 29, 0.8)',  # Red
        'Never Purchased': 'rgba(108, 117, 125, 0.8)'  # Gray
    }
    
    # Assign colors to nodes
    node_colors = []
    for node in nodes:
        # Extract segment name (remove year)
        segment_name = node.rsplit(' (', 1)[0]
        node_colors.append(segment_colors.get(segment_name, 'rgba(128, 128, 128, 0.8)'))
    
    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=nodes,
            color=node_colors
        ),
        link=dict(
            source=transitions_df['Source_idx'],
            target=transitions_df['Target_idx'],
            value=transitions_df['Count'],
            color='rgba(200, 200, 200, 0.4)'
        )
    )])
    
    fig.update_layout(
        title=title,
        font=dict(size=12),
        height=600,
        width=1200
    )
    
    return fig


# Create Sankey for each year pair
for year_pair in all_transitions_df['Year_Pair'].unique():
    transitions = all_transitions_df[all_transitions_df['Year_Pair'] == year_pair]
    
    # Filter out "Never Purchased" for cleaner visualization (optional)
    # Uncomment the line below to exclude "Never Purchased" segments
    # transitions = transitions[~transitions['Source'].str.contains('Never Purchased')]
    # transitions = transitions[~transitions['Target'].str.contains('Never Purchased')]
    
    fig = create_sankey_diagram(
        transitions,
        title=f"Customer Segment Flow: {year_pair}"
    )
    
    # Save as HTML
    output_path = f'../visualizations/sankey_{year_pair}.html'
    fig.write_html(output_path)
    print(f"Saved: {output_path}")
    
    # Display
    fig.show()

## 5. Multi-Year Sankey Diagram

Create a comprehensive Sankey showing all years in one diagram.

In [ ]:
# Combine all transitions into one Sankey
combined_transitions = all_transitions_df.groupby(['Source', 'Target'])['Count'].sum().reset_index()

# Filter to show only active segments (optional - remove to show all)
# Exclude "Never Purchased" for cleaner visualization
combined_transitions_filtered = combined_transitions[
    ~combined_transitions['Source'].str.contains('Never Purchased') &
    ~combined_transitions['Target'].str.contains('Never Purchased')
]

fig_combined = create_sankey_diagram(
    combined_transitions_filtered,
    title="Customer Segment Flow: Multi-Year Overview (2021-2025)"
)

# Save
fig_combined.write_html('../visualizations/sankey_multiyear.html')
print("Saved: ../visualizations/sankey_multiyear.html")

fig_combined.show()

## 6. Transition Analysis

Calculate key metrics about segment transitions.

In [ ]:
def analyze_segment_retention(df, year1, year2):
    """
    Analyze segment retention and migration patterns.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Yearly snapshot dataframe
    year1 : int
        Starting year
    year2 : int
        Ending year
        
    Returns:
    --------
    dict
        Dictionary with retention metrics
    """
    # Get transitions
    year1_data = df[df['YEAR'] == str(year1)][['Member_ID', 'Recency_Text']].copy()
    year2_data = df[df['YEAR'] == str(year2)][['Member_ID', 'Recency_Text']].copy()
    
    year1_data.columns = ['Member_ID', 'Segment_From']
    year2_data.columns = ['Member_ID', 'Segment_To']
    
    transitions = year1_data.merge(year2_data, on='Member_ID', how='inner')
    
    # Calculate retention rates by segment
    retention_rates = {}
    
    for segment in transitions['Segment_From'].unique():
        segment_data = transitions[transitions['Segment_From'] == segment]
        total = len(segment_data)
        retained = len(segment_data[segment_data['Segment_To'] == segment])
        
        retention_rate = (retained / total * 100) if total > 0 else 0
        retention_rates[segment] = {
            'total': total,
            'retained': retained,
            'retention_rate': retention_rate
        }
    
    return retention_rates


# Analyze retention for each year pair
print("\n=== SEGMENT RETENTION ANALYSIS ===")
print("=" * 80)

for i in range(len(years) - 1):
    year1, year2 = years[i], years[i + 1]
    print(f"\n{year1} → {year2}")
    print("-" * 80)
    
    retention = analyze_segment_retention(snapshot_df, year1, year2)
    
    retention_df = pd.DataFrame(retention).T
    retention_df = retention_df.sort_values('retention_rate', ascending=False)
    
    print(f"{'Segment':<20} {'Total':<10} {'Retained':<10} {'Retention Rate':<15}")
    print("-" * 80)
    
    for segment, row in retention_df.iterrows():
        print(f"{segment:<20} {int(row['total']):<10} {int(row['retained']):<10} {row['retention_rate']:.1f}%")

## 7. Segment Migration Patterns

Identify most common segment transitions.

In [ ]:
# Analyze common migration patterns
print("\n=== TOP SEGMENT MIGRATIONS ===")
print("=" * 80)

for year_pair in all_transitions_df['Year_Pair'].unique():
    transitions = all_transitions_df[all_transitions_df['Year_Pair'] == year_pair]
    
    # Remove self-transitions (segment retention)
    migrations = transitions[
        transitions['Source'].str.rsplit(' (', 1).str[0] != 
        transitions['Target'].str.rsplit(' (', 1).str[0]
    ]
    
    # Sort by count
    top_migrations = migrations.nlargest(10, 'Count')
    
    print(f"\n{year_pair}")
    print("-" * 80)
    print(f"{'From':<30} {'To':<30} {'Count':<10}")
    print("-" * 80)
    
    for _, row in top_migrations.iterrows():
        source = row['Source'].rsplit(' (', 1)[0]
        target = row['Target'].rsplit(' (', 1)[0]
        count = int(row['Count'])
        print(f"{source:<30} {target:<30} {count:<10}")

## 8. Churn and Reactivation Analysis

In [ ]:
# Analyze churn (Active/At Risk → Dormant/Inactive)
# and reactivation (Dormant/Inactive → Active)

def analyze_churn_reactivation(transitions_df):
    """
    Analyze churn and reactivation patterns.
    """
    # Define segment groups
    active_segments = ['Active', 'At Risk']
    inactive_segments = ['Inactive', 'Dormant', 'Never Purchased']
    
    results = []
    
    for year_pair in transitions_df['Year_Pair'].unique():
        pair_data = transitions_df[transitions_df['Year_Pair'] == year_pair]
        
        # Extract segment names (remove year)
        pair_data['Source_Segment'] = pair_data['Source'].str.rsplit(' (', 1).str[0]
        pair_data['Target_Segment'] = pair_data['Target'].str.rsplit(' (', 1).str[0]
        
        # Churn: Active → Inactive
        churn = pair_data[
            pair_data['Source_Segment'].isin(active_segments) &
            pair_data['Target_Segment'].isin(inactive_segments)
        ]['Count'].sum()
        
        # Reactivation: Inactive → Active
        reactivation = pair_data[
            pair_data['Source_Segment'].isin(inactive_segments) &
            pair_data['Target_Segment'].isin(active_segments)
        ]['Count'].sum()
        
        # Total active customers at start
        total_active_start = pair_data[
            pair_data['Source_Segment'].isin(active_segments)
        ]['Count'].sum()
        
        # Churn rate
        churn_rate = (churn / total_active_start * 100) if total_active_start > 0 else 0
        
        results.append({
            'Year_Pair': year_pair,
            'Churn_Count': churn,
            'Reactivation_Count': reactivation,
            'Net_Change': reactivation - churn,
            'Churn_Rate_%': churn_rate
        })
    
    return pd.DataFrame(results)


churn_reactivation_df = analyze_churn_reactivation(all_transitions_df)

print("\n=== CHURN AND REACTIVATION ANALYSIS ===")
print("=" * 80)
display(churn_reactivation_df)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(churn_reactivation_df))
width = 0.35

ax.bar(x - width/2, churn_reactivation_df['Churn_Count'], width, 
       label='Churn', color='#C73E1D', alpha=0.8)
ax.bar(x + width/2, churn_reactivation_df['Reactivation_Count'], width,
       label='Reactivation', color='#2E86AB', alpha=0.8)

ax.set_xlabel('Year Transition')
ax.set_ylabel('Number of Customers')
ax.set_title('Churn vs Reactivation by Year', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(churn_reactivation_df['Year_Pair'])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/churn_reactivation.png', dpi=300, bbox_inches='tight')
plt.show()

## Conclusion

### Key Insights:

1. **Segment Flow Patterns**: Sankey diagrams reveal how customers naturally transition between engagement states
2. **Retention Rates**: Active segment retention indicates health of customer relationships
3. **Churn Dynamics**: Understanding flow from Active → At Risk → Dormant helps target interventions
4. **Reactivation Opportunities**: Tracking Dormant → Active transitions shows win-back effectiveness

### Business Applications:

- **Predictive Interventions**: Target customers showing signs of downward segment movement
- **Win-Back Campaigns**: Focus on customers transitioning to Dormant/Inactive
- **Loyalty Programs**: Reward customers maintaining Active status
- **Resource Allocation**: Prioritize segments with highest revenue and retention potential

### Technical Achievement:

This analysis successfully replicates Power BI DAX functionality in Python:
- ✅ Yearly snapshot calculation (equivalent to Power BI ADDCOLUMNS)
- ✅ Cumulative metrics and time intelligence
- ✅ Segment classification (Recency_Text)
- ✅ Interactive Sankey visualizations
- ✅ Transition analysis and retention metrics

### Portfolio Value:

Demonstrates ability to:
- Translate business logic from BI tools to Python
- Create complex data transformations and calculations
- Build interactive visualizations for stakeholder communication
- Perform customer lifecycle analysis
- Generate actionable business insights